# FieldWing - Notebook 1: Maize leaf-disease model (train + TFLite export)

**Before running:** Settings -> Accelerator = GPU T4, Internet = ON, and add your maize dataset via *Add Input*.
Runs top to bottom in about 15-20 minutes. Outputs land in `/kaggle/working`.

In [ ]:
import os, json, random, time, pathlib
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

SEED, IMG, BATCH = 42, 224, 32
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
OUT = "/kaggle/working"
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Find the images (label = parent folder name)

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
rows = [(str(p), p.parent.name)
        for p in sorted(pathlib.Path("/kaggle/input").rglob("*")) if p.suffix.lower() in EXT]
df = pd.DataFrame(rows, columns=["path", "label"])
print(len(df), "images found")
print(df.label.value_counts())

# keep the 4 biggest folders (the 4 classes)
top = df.label.value_counts().head(4).index
df = df[df.label.isin(top)].reset_index(drop=True)
class_names = sorted(df.label.unique())
cls2id = {c: i for i, c in enumerate(class_names)}
print("\nClasses:", class_names)
assert len(class_names) == 4, "Expected 4 classes - check the folder listing above"

## 2. Stratified split 70 / 15 / 15

In [ ]:
train_df, tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=SEED)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d.label.value_counts().to_dict())

# quick look at the data
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, (_, r) in zip(ax.ravel(), train_df.sample(8, random_state=1).iterrows()):
    a.imshow(plt.imread(r.path)); a.set_title(r.label); a.axis("off")
plt.show()

## 3. Input pipeline (augmentation only on the training set)

In [ ]:
def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(img, tf.float32), label          # pixels stay in 0-255

aug = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
])

def make_ds(d, train):
    paths = d.path.values
    labels = d.label.map(cls2id).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    if train:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

cw = compute_class_weight("balanced", classes=np.arange(len(class_names)),
                          y=train_df.label.map(cls2id).values)
class_weight = dict(enumerate(cw))
print("class weights:", {class_names[k]: round(v, 2) for k, v in class_weight.items()})

## 4. Model: MobileNetV2 transfer learning

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input((IMG, IMG, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)        # 0-255 -> -1..1 (baked into the model)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cb = lambda: [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss")]

# Stage 1: train the new head only
h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight, callbacks=cb())

In [ ]:
# Stage 2: fine-tune the top of the backbone (BatchNorm layers stay frozen)
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight, callbacks=cb())

# training curves
acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
tr = h1.history["accuracy"] + h2.history["accuracy"]
plt.plot(tr, label="train acc"); plt.plot(acc, label="val acc")
plt.axvline(len(h1.history["accuracy"]) - 1, ls="--", c="gray"); plt.legend(); plt.title("Training")
plt.savefig(f"{OUT}/training_curves.png", dpi=150); plt.show()

## 5. Evaluation on the held-out test set

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
X_test = np.concatenate([x.numpy() for x, _ in test_ds])
probs = model.predict(X_test, batch_size=BATCH, verbose=0)
pred = probs.argmax(1)

print(classification_report(y_true, pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(class_names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); plt.tight_layout()
plt.savefig(f"{OUT}/confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Confidence-threshold sweep: how many healthy leaves would be wrongly flagged for spraying?
healthy_id = next(i for i, c in enumerate(class_names) if "health" in c.lower())
conf = probs.max(1)
rows = []
for t in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]:
    keep = conf >= t
    flagged_wrong = int(((pred != healthy_id) & (y_true == healthy_id) & keep).sum())
    rows.append(dict(threshold=t,
                     coverage=round(float(keep.mean()), 3),
                     accuracy_on_kept=round(float((pred[keep] == y_true[keep]).mean()), 4) if keep.any() else None,
                     healthy_wrongly_flagged=flagged_wrong,
                     healthy_total=int((y_true == healthy_id).sum())))
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))
sweep.to_csv(f"{OUT}/threshold_sweep.csv", index=False)

## 6. Save + export to TFLite (float32 and dynamic-range quantized)

In [ ]:
model.save(f"{OUT}/maize_model.keras")
json.dump(class_names, open(f"{OUT}/class_names.json", "w"))
test_df.to_csv(f"{OUT}/test_split.csv", index=False)

conv = tf.lite.TFLiteConverter.from_keras_model(model)
fp32 = conv.convert()
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
dyn = conv.convert()

open(f"{OUT}/maize_model_fp32.tflite", "wb").write(fp32)
open(f"{OUT}/maize_model_dynamic.tflite", "wb").write(dyn)

def run_tflite(blob, X):
    it = tf.lite.Interpreter(model_content=blob, num_threads=1)
    it.allocate_tensors()
    i_idx, o_idx = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
    out = []
    for x in X:
        it.set_tensor(i_idx, x[None].astype(np.float32)); it.invoke()
        out.append(it.get_tensor(o_idx)[0])
    return np.array(out)

results = {}
for name, blob in [("keras_fp32", None), ("tflite_fp32", fp32), ("tflite_dynamic", dyn)]:
    if blob is None:
        p, size, lat = probs, os.path.getsize(f"{OUT}/maize_model.keras") / 1e6, None
    else:
        t0 = time.time(); run_tflite(blob, X_test[:100]); lat = (time.time() - t0) / 100 * 1000
        p, size = run_tflite(blob, X_test), len(blob) / 1e6
    results[name] = dict(size_mb=round(size, 2),
                         test_accuracy=round(float((p.argmax(1) == y_true).mean()), 4),
                         cpu_ms_per_image=None if lat is None else round(lat, 1))
res = pd.DataFrame(results).T
print(res)
res.to_csv(f"{OUT}/model_comparison.csv")
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)

## Done

Download from the *Output* tab (or use **Save Version -> Save & Run All**, then reuse the output in Notebook 2):
`maize_model_dynamic.tflite`, `maize_model_fp32.tflite`, `class_names.json`, `test_split.csv`, plus the plots and CSVs.
Input to the TFLite models: float32, shape `(1, 224, 224, 3)`, **pixel values 0-255** (scaling is inside the model).